In [1]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

A decision tree asks a series of yes/no questions to split data into purer and purer groups. "Is Sex == male?" then "Is Age < 10?" etc. — each split tries to separate survivors from non-survivors as cleanly as possible.

In [2]:
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

# Minimal preprocessing
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df['Embarked'] = df['Embarked'].fillna('S').map({'S': 0, 'C': 1, 'Q': 2})

features = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked']
X = df[features]
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Single decision tree — intentionally unlimited depth to show overfitting
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)

train_acc = accuracy_score(y_train, tree.predict(X_train))
test_acc = accuracy_score(y_test, tree.predict(X_test))
print(f"Train accuracy: {train_acc:.3f}")
print(f"Test accuracy: {test_acc:.3f}")

Train accuracy: 0.978
Test accuracy: 0.777


This gap is the whole reason ensembles exist. A single unlimited-depth tree fits training data almost perfectly but generalizes poorly. Everything in the rest of today's session is about fixing exactly this problem.

Random Forest's fix: don't trust one tree, trust many imperfect trees averaged together. Each tree sees a random subset of rows (bootstrap sampling) and a random subset of columns at each split — this is called bagging.

In [3]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,     # 100 trees, each trained independently
    max_depth=6,           # limit depth — prevents each tree from overfitting too much
    random_state=42
)
rf.fit(X_train, y_train)

train_acc = accuracy_score(y_train, rf.predict(X_train))
test_acc = accuracy_score(y_test, rf.predict(X_test))
print(f"RF Train accuracy: {train_acc:.3f}")
print(f"RF Test accuracy: {test_acc:.3f}")

# Feature importance — which columns mattered most?
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("\nFeature importances:")
print(importances)

RF Train accuracy: 0.876
RF Test accuracy: 0.827

Feature importances:
Sex         0.438646
Fare        0.230238
Age         0.159475
Pclass      0.135973
Embarked    0.035668
dtype: float64


XGBoost's fix: instead of training trees independently, train them one after another, each correcting the previous tree's mistakes. This is boosting — usually higher accuracy than bagging, but needs more careful tuning to avoid overfitting.

Random Forest — bagging
Trees trained independently, in parallel. Final answer = average vote. Reduces variance.

XGBoost — boosting
Trees trained sequentially. Each new tree fixes previous errors. Reduces bias.

In [4]:
!pip install xgboost -q

from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=100,
    max_depth=4,           # shallower trees than RF — boosting needs weaker individual trees
    learning_rate=0.1,     # how much each new tree corrects the previous ensemble
    eval_metric='logloss',
    random_state=42
)
xgb.fit(X_train, y_train)

train_acc = accuracy_score(y_train, xgb.predict(X_train))
test_acc = accuracy_score(y_test, xgb.predict(X_test))
print(f"XGBoost Train accuracy: {train_acc:.3f}")
print(f"XGBoost Test accuracy: {test_acc:.3f}")

XGBoost Train accuracy: 0.897
XGBoost Test accuracy: 0.799


Now tune properly with GridSearchCV and compare all 3 models fairly. Hand-picked hyperparameters (like max_depth=6 above) are guesses — GridSearchCV systematically searches for the best combination.

In [5]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6, 8],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV accuracy:", grid_search.best_score_)

best_rf = grid_search.best_estimator_
test_acc = accuracy_score(y_test, best_rf.predict(X_test))
print("Tuned RF test accuracy:", test_acc)

Best params: {'max_depth': 8, 'min_samples_split': 2, 'n_estimators': 200}
Best CV accuracy: 0.8441248891953117
Tuned RF test accuracy: 0.8379888268156425


In [6]:
# Final comparison table across everything trained today
results = {
    'Single Tree': accuracy_score(y_test, tree.predict(X_test)),
    'Random Forest (default)': accuracy_score(y_test, rf.predict(X_test)),
    'Random Forest (tuned)': test_acc,
    'XGBoost': accuracy_score(y_test, xgb.predict(X_test))
}
for name, acc in results.items():
    print(f"{name}: {acc:.3f}")

Single Tree: 0.777
Random Forest (default): 0.827
Random Forest (tuned): 0.838
XGBoost: 0.799
